# Colab Baseline Runner (Costa)

Run Costa baseline modeling on Colab with Google Drive feature artifacts and DagsHub/MLflow tracking.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!if [ -d /content/PFE_Experiments/.git ]; then git -C /content/PFE_Experiments pull; else git clone https://github.com/aminetech26/PFE_Experiments.git; fi
%cd /content/PFE_Experiments

In [ ]:
%pip install -q uv
!uv sync

In [ ]:
# Set these for your session (or use Colab secrets)
import os
os.environ['DAGSHUB_USERNAME'] = 'aminetech26'
os.environ['DAGSHUB_REPO'] = 'PFE_Experiments'
os.environ['DAGSHUB_USER_TOKEN'] = '5e31845f92874871e830dd2f59859e3d632c1aa0'

In [ ]:
# Initialize Drive folders used by Optuna persistence
!uv run python -m src.training.colab_drive_init --init

In [ ]:
# Redirect all experiment writes to Google Drive (persistent across sessions)
import os
os.environ['PFE_ARTIFACTS_ROOT'] = '/content/drive/MyDrive/PV-FDD/optuna/artifacts'
print('PFE_ARTIFACTS_ROOT =', os.environ['PFE_ARTIFACTS_ROOT'])

## Drive-first feature materialization

Copy precomputed feature artifacts from Google Drive into the repo workspace.

Expected Drive source: `MyDrive/PV-FDD/data/features/costa/`

In [ ]:
from pathlib import Path
import shutil

drive_features = Path('/content/drive/MyDrive/PV-FDD/data/features/costa')
repo_features = Path('/content/PFE_Experiments/data/processed/features/costa')

if not drive_features.exists():
    raise FileNotFoundError(f'Drive feature directory not found: {drive_features}')

repo_features.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(drive_features, repo_features, dirs_exist_ok=True)
print(f'Copied Costa feature artifacts to: {repo_features}')

In [ ]:
print('Using precomputed Costa feature runs from Google Drive.')

In [ ]:
import os
os.environ['MPLBACKEND'] = 'Agg'

## Baseline Commands (per task)

Use one command per task. For each new run, update your choices in `configs/model_config.yaml` (active model, seed, HPO) and ensure the requested `task/profile/split_path` already exists in the generated feature artifacts.

In [ ]:
!uv run python -m src.modeling.anomaly_detection.ml.run --model one_class_svm --task anomaly_semisup --dataset costa --split-path path_a --profile baseline_raw --run-type baseline --seed 42 --kernel rbf